[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_1/03_tag_1_klassifikation_metriken.ipynb)

# Tag 1 - 03 Klassifikation, Verteilungen, ROC und Confusion Matrix

Klassifikation sagt Kategorien vorher. Dieses Notebook zeigt wieder 1D-Verteilungen, 2D-Feature-Kombinationen, Thresholds, ROC-Kurve und Confusion Matrix.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True
print('Setup abgeschlossen.')


In [ ]:
def make_quality_data(seed=RANDOM_STATE, overlap=0.45, imbalance=0.30):
    local_rng = np.random.default_rng(seed)
    n_total = 260
    n_pos = int(n_total * imbalance)
    n_neg = n_total - n_pos
    ok = local_rng.normal(loc=[0.25, 0.35], scale=[0.20 + overlap * 0.12, 0.22 + overlap * 0.12], size=(n_neg, 2))
    defect = local_rng.normal(loc=[0.95, 0.95], scale=[0.20 + overlap * 0.15, 0.22 + overlap * 0.15], size=(n_pos, 2))
    X = np.vstack([ok, defect])
    y = np.r_[np.zeros(n_neg, dtype=int), np.ones(n_pos, dtype=int)]
    order = local_rng.permutation(len(y))
    return pd.DataFrame(X[order], columns=['Vibration', 'Temperaturabweichung']).assign(Fehlerhaft=y[order])

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -40, 40)))

def fit_logistic(X, y, learning_rate=0.12, steps=5000):
    mean = X.mean(axis=0)
    std = np.where(X.std(axis=0) == 0, 1, X.std(axis=0))
    Xs = (X - mean) / std
    Xb = np.c_[np.ones(len(Xs)), Xs]
    weights = np.zeros(Xb.shape[1])
    for _ in range(steps):
        score = Xb @ weights
        proba = sigmoid(score)
        weights -= learning_rate * (Xb.T @ (proba - y) / len(y))
    return weights, mean, std

def predict_score(X, weights, mean, std):
    Xs = (np.asarray(X, dtype=float) - mean) / std
    return np.c_[np.ones(len(Xs)), Xs] @ weights

def predict_proba(X, weights, mean, std):
    return sigmoid(predict_score(X, weights, mean, std))

def confusion_and_metrics_from_score(y_true, score, score_threshold):
    pred = (score >= score_threshold).astype(int)
    tp = int(np.sum((pred == 1) & (y_true == 1)))
    tn = int(np.sum((pred == 0) & (y_true == 0)))
    fp = int(np.sum((pred == 1) & (y_true == 0)))
    fn = int(np.sum((pred == 0) & (y_true == 1)))
    accuracy = (tp + tn) / len(y_true)
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    return pred, np.array([[tn, fp], [fn, tp]]), accuracy, precision, recall, f1

def roc_points(y_true, proba):
    thresholds = np.linspace(0, 1, 101)
    rows = []
    for threshold in thresholds:
        pred = (proba >= threshold).astype(int)
        tp = int(np.sum((pred == 1) & (y_true == 1)))
        tn = int(np.sum((pred == 0) & (y_true == 0)))
        fp = int(np.sum((pred == 1) & (y_true == 0)))
        fn = int(np.sum((pred == 0) & (y_true == 1)))
        rows.append((threshold, fp / max(1, fp + tn), tp / max(1, tp + fn)))
    return pd.DataFrame(rows, columns=['probability_threshold', 'fpr', 'tpr'])


## Interaktiv: Verteilungen, Entscheidungsgrenze, ROC und Confusion Matrix


In [ ]:
def classification_demo(feature='Vibration', score_threshold=0.0, overlap=0.45, positive_klasse_prozent=30):
    df = make_quality_data(overlap=overlap, imbalance=positive_klasse_prozent / 100)
    X = df[['Vibration', 'Temperaturabweichung']].to_numpy()
    y = df['Fehlerhaft'].to_numpy()
    ids = np.random.default_rng(RANDOM_STATE).permutation(len(y))
    test_size = int(0.25 * len(y))
    test_idx, train_idx = ids[:test_size], ids[test_size:]
    weights, mean, std = fit_logistic(X[train_idx], y[train_idx])
    score_test = predict_score(X[test_idx], weights, mean, std)
    proba_test = sigmoid(score_test)
    pred, matrix, accuracy, precision, recall, f1 = confusion_and_metrics_from_score(y[test_idx], score_test, score_threshold)
    roc_df = roc_points(y[test_idx], proba_test)

    xx, yy = np.meshgrid(
        np.linspace(df['Vibration'].min() - 0.25, df['Vibration'].max() + 0.25, 180),
        np.linspace(df['Temperaturabweichung'].min() - 0.25, df['Temperaturabweichung'].max() + 0.25, 180),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    score_grid = predict_score(grid, weights, mean, std).reshape(xx.shape)
    proba_equivalent = sigmoid(score_threshold)

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    for label, name, color in [(0, 'OK', '#4c78a8'), (1, 'Fehlerhaft', '#e45756')]:
        part = df[df['Fehlerhaft'] == label]
        axes[0, 0].hist(part[feature], bins=22, alpha=0.55, label=name, color=color)
    axes[0, 0].set_title(f'1D-Verteilung: {feature}')
    axes[0, 0].legend()

    score_min, score_max = np.nanmin(score_grid), np.nanmax(score_grid)
    if score_min < score_threshold < score_max:
        axes[0, 1].contour(xx, yy, score_grid, levels=[score_threshold], colors='black', linewidths=2)
    axes[0, 1].contourf(xx, yy, score_grid, levels=30, cmap='RdBu_r', alpha=0.25)
    for label, name, color in [(0, 'OK', '#4c78a8'), (1, 'Fehlerhaft', '#e45756')]:
        part = df[df['Fehlerhaft'] == label]
        axes[0, 1].scatter(part['Vibration'], part['Temperaturabweichung'], alpha=0.65, label=name, color=color)
    axes[0, 1].set_xlabel('Vibration')
    axes[0, 1].set_ylabel('Temperaturabweichung')
    axes[0, 1].set_title(f'Grenze: Score={score_threshold:.1f} | entspricht P={proba_equivalent:.2f}')
    axes[0, 1].legend()

    axes[1, 0].imshow(matrix, cmap='Blues')
    axes[1, 0].set_xticks([0, 1], ['vorh. OK', 'vorh. Fehler'])
    axes[1, 0].set_yticks([0, 1], ['ist OK', 'ist Fehler'])
    for row in range(2):
        for col in range(2):
            axes[1, 0].text(col, row, matrix[row, col], ha='center', va='center', fontsize=14)
    axes[1, 0].set_title(f'Acc={accuracy:.2f}, Prec={precision:.2f}, Rec={recall:.2f}, F1={f1:.2f}')

    axes[1, 1].plot(roc_df['fpr'], roc_df['tpr'], label='ROC')
    tn, fp, fn, tp = matrix.ravel()
    current_fpr = fp / max(1, fp + tn)
    current_tpr = tp / max(1, tp + fn)
    axes[1, 1].scatter([current_fpr], [current_tpr], color='red', s=80, label='aktueller Score-Grenzwert')
    axes[1, 1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
    axes[1, 1].set_xlabel('False Positive Rate')
    axes[1, 1].set_ylabel('True Positive Rate / Recall')
    axes[1, 1].set_title('ROC-Kurve')
    axes[1, 1].legend()
    plt.tight_layout()
    plt.show()

widgets.interact(
    classification_demo,
    feature=widgets.Dropdown(options=['Vibration', 'Temperaturabweichung'], value='Vibration'),
    score_threshold=widgets.FloatSlider(value=0.0, min=-8.0, max=8.0, step=0.2),
    overlap=widgets.FloatSlider(value=0.45, min=0.0, max=1.0, step=0.05),
    positive_klasse_prozent=widgets.IntSlider(value=30, min=5, max=50, step=5),
);


## Klassenungleichgewicht

Wenn eine Klasse selten ist, kann Accuracy gut aussehen, obwohl das Modell die seltene Klasse kaum findet. Deshalb sind Recall und Precision wichtig.


In [ ]:
def imbalanced_demo(positive_klasse_prozent=8, score_threshold=0.0, overlap=0.65):
    df = make_quality_data(overlap=overlap, imbalance=positive_klasse_prozent / 100)
    X = df[['Vibration', 'Temperaturabweichung']].to_numpy()
    y = df['Fehlerhaft'].to_numpy()
    ids = np.random.default_rng(RANDOM_STATE).permutation(len(y))
    test_size = int(0.30 * len(y))
    test_idx, train_idx = ids[:test_size], ids[test_size:]
    weights, mean, std = fit_logistic(X[train_idx], y[train_idx])
    score_test = predict_score(X[test_idx], weights, mean, std)
    _, matrix, accuracy, precision, recall, f1 = confusion_and_metrics_from_score(y[test_idx], score_test, score_threshold)
    majority_pred = np.zeros_like(y[test_idx])
    majority_accuracy = np.mean(majority_pred == y[test_idx])
    majority_recall = 0.0

    display(pd.DataFrame({
        'Modell': ['immer OK', 'logistische Regression'],
        'Accuracy': [majority_accuracy, accuracy],
        'Precision Fehlerhaft': [0.0, precision],
        'Recall Fehlerhaft': [majority_recall, recall],
        'F1 Fehlerhaft': [0.0, f1],
    }).round(3))

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    counts = pd.Series(y[test_idx]).value_counts().sort_index()
    axes[0].bar(['OK', 'Fehlerhaft'], [counts.get(0, 0), counts.get(1, 0)], color=['#4c78a8', '#e45756'])
    axes[0].set_title('Testdaten: Klassenverteilung')
    axes[0].set_ylabel('Samples')

    axes[1].imshow(matrix, cmap='Blues')
    axes[1].set_xticks([0, 1], ['vorh. OK', 'vorh. Fehler'])
    axes[1].set_yticks([0, 1], ['ist OK', 'ist Fehler'])
    for row in range(2):
        for col in range(2):
            axes[1].text(col, row, matrix[row, col], ha='center', va='center', fontsize=14)
    axes[1].set_title(f'Accuracy={accuracy:.2f}, Recall={recall:.2f}')
    plt.tight_layout()
    plt.show()

widgets.interact(
    imbalanced_demo,
    positive_klasse_prozent=widgets.IntSlider(value=8, min=2, max=40, step=2),
    score_threshold=widgets.FloatSlider(value=0.0, min=-8.0, max=8.0, step=0.2),
    overlap=widgets.FloatSlider(value=0.65, min=0.0, max=1.0, step=0.05),
);
